# 🌍 **Reasoning Over Earth Engine**
## **An AI Agent for Kenya's Climate Risk**

An open replica of the **Google I/O 2026 Geospatial Reasoning** pattern, 
running on **Gemini 3.5 Flash** and the free Earth Engine tier.

Ask a plain-English question about climate risk in Kenya and get a 
grounded, ranked, explainable answer. Run the cells top to bottom.

> All logic lives in `src/`; this notebook just imports and calls it.

## **Install dependencies**

In [ ]:
# If running in Colab, clone the repo first (uncomment and set your URL):
# !git clone https://github.com/<your-org>/kenya-climate-agent.git
# %cd kenya-climate-agent

!pip install -q -r requirements.txt
print('Dependencies installed.')

## **Authenticate Earth Engine**

In Colab this opens an auth link. Set your project id below. Without it, 
the agent still runs using deterministic fallback values.

In [ ]:
import os

# Set your credentials (or rely on a .env file when running locally).
os.environ.setdefault('GOOGLE_API_KEY', 'your-gemini-api-key-here')
os.environ.setdefault('EE_PROJECT_ID', 'your-earthengine-project-id-here')

try:
    import ee
    ee.Authenticate()           # opens an auth flow in Colab
    ee.Initialize(project=os.environ['EE_PROJECT_ID'])
    print('Earth Engine initialized.')
except Exception as exc:
    print(f'Earth Engine not initialized ({exc}). Running in fallback mode.')

## **Build the RAG index**

Loads the documents in `knowledge/` (markdown, notes, and any PDFs you 
added), chunks and embeds them, and persists a searchable index.

In [ ]:
from src.rag.build_index import build_index

summary = build_index()
print(f"Knowledge index ready: {summary['count']} passages "
      f"(mode={summary['mode']}, dim={summary['dim']}).")

## **Run the agent**

Ask a question. The agent plans (Gemini 3.5 Flash), calls the tools, and 
fuses a grounded answer.

In [ ]:
from src.agent.orchestrator import run_agent

question = (
    'Which counties in Kenya face the highest flood risk this season, '
    'and who is most exposed?'
)

result = run_agent(question)
print('Success:', result.success)
print('Tools run:', [s.tool for s in result.step_results])

## **Display the results**

In [ ]:
from src.utils import viz

# The grounded natural-language answer
print(result.answer)
print('\n' + '=' * 70 + '\n')

# The ranked county table
ranking = result.collected.get('compute_risk_score', {}).get('ranking', [])
df = viz.risk_dataframe(ranking)
if df is not None and not df.empty:
    display(df[['county', 'risk_score', 'exposed_population']])
else:
    print('No ranking produced.')

## **Visualize the map**

An interactive county risk map plus the risk bar chart.

In [ ]:
# Risk bar chart
fig = viz.build_risk_bar_chart(ranking, top_n=10)
if fig is not None:
    fig


In [ ]:
# Interactive risk map (folium). Renders inline in the notebook.
risk_map = viz.build_risk_map(ranking, top_n=15)
risk_map  # display the map


---
**Note:** This is an open replica of the I/O 2026 Geospatial Reasoning 
pattern — not Google's gated agent. Without a Gemini key or Earth Engine 
auth it uses deterministic fallback values that are not real measurements. 
Validate against official sources (Kenya Met, NDMA) before operational use.